# 항법 필터 설계 (Navigation Filter Design)

---

## 왜 항법 필터가 필요한가?

> **"유도탄의 자기 위치/속도를 정확히 아는 것이 유도 성능의 기본"**

비례항법(PN), 최적 유도(OGL), MPC 등 어떤 유도법칙을 쓰더라도, 그 입력으로 들어가는 **자기 위치·속도·자세**가 정확하지 않으면 유도 성능은 크게 저하됩니다. 예를 들어 위치 오차가 10m이면 비례항법의 명령에 직접 bias가 생기고, Miss Distance가 증가합니다.

**항법 필터의 역할:**
- IMU (Gyro + Accelerometer) 측정값의 오차(bias, 잡음)를 보정
- GPS, Radar 등 외부 측정값과 융합하여 최적 추정치 산출
- 불확실도(covariance)를 정량화하여 유도/제어계에 품질 정보 제공

---

## 학습 목표

1. **Kalman Filter 기초** — Predict-Update 사이클, 최적 추정의 의미 (Bar-Shalom Ch.5)
2. **Alpha-Beta / Alpha-Beta-Gamma 필터** — 정상상태 Kalman의 추적 필터 응용
3. **Strapdown INS 기계화** — 쿼터니언 적분, 속도/위치 갱신 (Siouris Ch.3)
4. **GPS 측정 모델** — 위치/속도 측정 잡음, 업데이트 주기
5. **GPS-Aided INS (15-State Error-State EKF)** — INS + GPS 통합 항법 (Bar-Shalom Ch.6)
6. **Covariance 분석** — 필터 성능의 정량적 평가

---

**참고문헌:**
- Bar-Shalom, Y. et al., *Estimation with Applications to Tracking and Navigation*, Wiley, 2001. Chs. 5–6.
- Siouris, G.M., *Missile Guidance and Control Systems*, Springer, 2004. Ch. 3 (INS mechanization).
- Groves, P.D., *Principles of GNSS, Inertial, and Multisensor Integrated Navigation Systems*, 2nd ed., Artech House, 2013.
- Titterton & Weston, *Strapdown Inertial Navigation Technology*, 2nd ed., IET, 2004.

## 1. 칼만 필터 기초 (Kalman Filter Basics)

Bar-Shalom Ch. 5에 따르면, Kalman Filter는 **선형 가우시안 시스템에서 최소 분산 비편향 추정기 (MVUE)**입니다.

---

### 시스템 모델

$$
\mathbf{x}_{k} = \mathbf{F}\,\mathbf{x}_{k-1} + \mathbf{w}_{k-1}, \quad \mathbf{w} \sim \mathcal{N}(\mathbf{0},\,\mathbf{Q})
$$

$$
\mathbf{z}_{k} = \mathbf{H}\,\mathbf{x}_{k} + \mathbf{v}_{k}, \quad \mathbf{v} \sim \mathcal{N}(\mathbf{0},\,\mathbf{R})
$$

- $\mathbf{x}$: 상태 벡터 (위치, 속도, ...)
- $\mathbf{F}$: 상태 전이 행렬 (State Transition Matrix)
- $\mathbf{Q}$: 프로세스 잡음 공분산 (Process Noise Covariance)
- $\mathbf{z}$: 측정 벡터
- $\mathbf{H}$: 측정 행렬 (Measurement Matrix)
- $\mathbf{R}$: 측정 잡음 공분산 (Measurement Noise Covariance)

---

### Predict-Update 사이클

#### Prediction (사전 추정, Time Update)

$$
\hat{\mathbf{x}}^{-}_{k} = \mathbf{F}\,\hat{\mathbf{x}}_{k-1}
$$

$$
\mathbf{P}^{-}_{k} = \mathbf{F}\,\mathbf{P}_{k-1}\,\mathbf{F}^\top + \mathbf{Q}
$$

#### Update (사후 추정, Measurement Update)

Innovation (잔차):
$$
\tilde{\mathbf{z}}_k = \mathbf{z}_k - \mathbf{H}\,\hat{\mathbf{x}}^{-}_k
$$

Innovation Covariance:
$$
\mathbf{S}_k = \mathbf{H}\,\mathbf{P}^{-}_k\,\mathbf{H}^\top + \mathbf{R}
$$

Kalman Gain (최적 게인):
$$
\mathbf{K}_k = \mathbf{P}^{-}_k\,\mathbf{H}^\top\,\mathbf{S}_k^{-1}
$$

상태 갱신:
$$
\hat{\mathbf{x}}_k = \hat{\mathbf{x}}^{-}_k + \mathbf{K}_k\,\tilde{\mathbf{z}}_k
$$

공분산 갱신 (Joseph form — 수치 안정성):
$$
\mathbf{P}_k = (\mathbf{I} - \mathbf{K}_k\mathbf{H})\,\mathbf{P}^{-}_k\,(\mathbf{I} - \mathbf{K}_k\mathbf{H})^\top + \mathbf{K}_k\,\mathbf{R}\,\mathbf{K}_k^\top
$$

> **Note:** 단순 형태 $\mathbf{P} = (\mathbf{I}-\mathbf{K}\mathbf{H})\mathbf{P}^-$는 $\mathbf{K}$가 최적일 때만 올바릅니다. Joseph form은 $\mathbf{K}$가 부최적이거나 수치오차가 있을 때도 $\mathbf{P}$의 양의 정부호성(PSD)을 보장합니다. (Bar-Shalom eq. 5.2.3.24)

---

### Kalman Gain의 물리적 의미

$$
\mathbf{K} = \frac{\mathbf{P}^{-}\mathbf{H}^\top}{\mathbf{H}\mathbf{P}^{-}\mathbf{H}^\top + \mathbf{R}}
\quad\text{(스칼라 경우)}
$$

- $\mathbf{R} \to 0$: 측정을 완전 신뢰 → $K \to 1/H$, 측정값으로 상태 대체
- $\mathbf{P}^- \to 0$: 예측을 완전 신뢰 → $K \to 0$, 측정값 무시
- 실전: $Q$와 $R$의 적절한 tuning이 필터 성능을 결정

In [ ]:
"""
1D Kalman Filter 예제: 자유낙하 물체 위치 추정

상태: x = [위치(m), 속도(m/s)]^T
측정: z = 위치 (잡음 포함)

Bar-Shalom Ch.5 기반 구현
"""
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

# ── 시뮬레이션 파라미터 ──────────────────────────────────────────────────────
dt      = 0.1          # 시간 간격 (s)
T_total = 10.0         # 총 시간 (s)
N       = int(T_total / dt)
g       = 9.81         # 중력가속도 (m/s^2)

# 측정 잡음 표준편차 (레이더 위치 측정)
sigma_meas = 5.0       # m

# ── 시스템 행렬 (등속-등가속 모델) ──────────────────────────────────────────
# 상태: [위치, 속도]
F = np.array([[1.0, dt],
              [0.0, 1.0]])

H = np.array([[1.0, 0.0]])   # 위치만 측정

# 프로세스 잡음 (가속도 불확실성 모델링)
# Q = q * G * G^T,  G = [dt^2/2, dt]^T
q_accel = 0.5   # m/s^2 — 모델에 없는 가속도 성분의 std
G = np.array([[0.5 * dt**2],
              [dt]])
Q = q_accel**2 * (G @ G.T)

R = np.array([[sigma_meas**2]])   # 측정 잡음 공분산

# ── 참값 생성 (자유낙하) ───────────────────────────────────────────────────
t_arr    = np.arange(N) * dt
pos_true = 100.0 - 0.5 * g * t_arr**2   # 초기 고도 100m에서 낙하
vel_true = -g * t_arr

# 측정값 생성
z_arr = pos_true + np.random.randn(N) * sigma_meas

# ── Kalman Filter 초기화 ───────────────────────────────────────────────────
x_est = np.array([z_arr[0], 0.0])  # 초기 추정: 첫 측정으로 위치, 속도=0
P_est = np.diag([sigma_meas**2, 100.0**2])  # 초기 공분산 (큰 불확실도)

pos_kf  = np.zeros(N)
vel_kf  = np.zeros(N)
std_kf  = np.zeros(N)   # 위치 1-sigma

pos_kf[0]  = x_est[0]
vel_kf[0]  = x_est[1]
std_kf[0]  = np.sqrt(P_est[0, 0])

# ── 필터 루프 ─────────────────────────────────────────────────────────────
for k in range(1, N):
    # 중력 입력 (알려진 입력)
    u = np.array([-g * dt])   # 속도 증분
    B = np.array([[0.0], [1.0]])

    # ── Predict ──
    x_pred = F @ x_est + B @ u
    P_pred = F @ P_est @ F.T + Q

    # ── Update ──
    z_k   = z_arr[k:k+1]   # 측정값 (shape [1])
    inn   = z_k - H @ x_pred
    S     = H @ P_pred @ H.T + R
    K     = P_pred @ H.T @ np.linalg.inv(S)   # Kalman gain [2x1]

    x_est = x_pred + (K @ inn).ravel()

    # Joseph form (수치 안정성)
    I   = np.eye(2)
    IKH = I - K @ H
    P_est = IKH @ P_pred @ IKH.T + K @ R @ K.T
    P_est = 0.5 * (P_est + P_est.T)   # 대칭성 강제

    pos_kf[k] = x_est[0]
    vel_kf[k] = x_est[1]
    std_kf[k] = np.sqrt(P_est[0, 0])

# ── 시각화 ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
fig.suptitle('1D Kalman Filter: 자유낙하 물체 위치/속도 추정', fontsize=14, fontweight='bold')

# 위치 플롯
ax1 = axes[0]
ax1.plot(t_arr, pos_true, 'k-', linewidth=2, label='참값 (True)')
ax1.scatter(t_arr, z_arr, s=8, c='gray', alpha=0.5, label=f'측정값 (σ={sigma_meas}m)')
ax1.plot(t_arr, pos_kf, 'r-', linewidth=2, label='Kalman 추정치')
ax1.fill_between(t_arr,
                 pos_kf - 3*std_kf,
                 pos_kf + 3*std_kf,
                 alpha=0.2, color='red', label='±3σ 범위')
ax1.set_xlabel('시간 (s)')
ax1.set_ylabel('위치 (m)')
ax1.set_title('위치 추정')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 속도 플롯
ax2 = axes[1]
ax2.plot(t_arr, vel_true, 'k-', linewidth=2, label='참값 (True)')
ax2.plot(t_arr, vel_kf, 'b-', linewidth=2, label='Kalman 추정치')
ax2.set_xlabel('시간 (s)')
ax2.set_ylabel('속도 (m/s)')
ax2.set_title('속도 추정 (측정 없이 Kalman이 추론)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 성능 지표
rmse_meas = np.sqrt(np.mean((z_arr - pos_true)**2))
rmse_kf   = np.sqrt(np.mean((pos_kf - pos_true)**2))
print(f'\n=== 성능 비교 ===')
print(f'측정값 RMSE:       {rmse_meas:.2f} m')
print(f'Kalman 추정 RMSE:  {rmse_kf:.2f} m')
print(f'잡음 감소율:        {(1 - rmse_kf/rmse_meas)*100:.1f}%')

## 2. Alpha-Beta / Alpha-Beta-Gamma 필터

### 정상상태 Kalman → 고정게인 추적 필터

기동이 없는 일정 운동 상황에서 Kalman Filter가 정상상태(steady-state)에 수렴하면, Kalman Gain **K**는 상수가 됩니다. 이를 고정 게인(fixed-gain)으로 미리 설정한 것이 **α-β 필터**입니다.

---

### α-β 필터 (위치 + 속도)

**Predict:**
$$
\hat{x}^- = \hat{x} + \hat{v} \cdot \Delta t, \qquad \hat{v}^- = \hat{v}
$$

**Update** (innovation $\nu = z - \hat{x}^-$):
$$
\hat{x} = \hat{x}^- + \alpha\,\nu, \qquad \hat{v} = \hat{v}^- + \frac{\beta}{\Delta t}\,\nu
$$

---

### α-β-γ 필터 (위치 + 속도 + 가속도)

기동 표적에 대응하기 위해 가속도 상태를 추가합니다.

**Predict:**
$$
\hat{x}^- = \hat{x} + \hat{v}\Delta t + \tfrac{1}{2}\hat{a}\Delta t^2
$$
$$
\hat{v}^- = \hat{v} + \hat{a}\Delta t, \qquad \hat{a}^- = \hat{a}
$$

**Update:**
$$
\hat{x} = \hat{x}^- + \alpha\,\nu
$$
$$
\hat{v} = \hat{v}^- + \frac{\beta}{\Delta t}\,\nu
$$
$$
\hat{a} = \hat{a}^- + \frac{2\gamma}{\Delta t^2}\,\nu
$$

---

### 유도탄에서의 응용

| 용도 | 필터 종류 | 이유 |
|------|-----------|------|
| Seeker LOS rate 추정 | α-β | 각도+각속도, 단순·빠름 |
| 기동 표적 각속도 추정 | α-β-γ | 가속도 성분 포함 |
| 고정밀 위치 추적 | Kalman EKF | 비선형 측정 모델 |

> **실전 팁:** α-β-γ에서 γ가 너무 크면 잡음에 민감해지고, 너무 작으면 기동을 놓칩니다. 통상 $\gamma \approx \beta^2 / (2\alpha)$ 관계(Benedict-Bordner condition)로 trade-off를 최적화합니다.

In [ ]:
"""
Alpha-Beta-Gamma 필터: 기동 표적의 LOS 각도/각속도/각가속도 추정

시나리오: 시커가 바라보는 표적의 LOS 각도를 측정
- 표적이 중간에 기동(횡 가속도 변화)
- α-β-γ 필터로 각도, 각속도, 각가속도 추정
"""
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)

dt      = 0.02    # 50 Hz 시커 (s)
T_total = 5.0
N       = int(T_total / dt)
t_arr   = np.arange(N) * dt

# ── 참값 생성: LOS 각도 (rad) ─────────────────────────────────────────────
# 표적이 등속 비행 후 t=2s부터 횡 가속도 발생 → LOS 각도 비선형 변화
lam_dot_true = np.zeros(N)
lam_ddot_true = np.zeros(N)

for k in range(N):
    t = t_arr[k]
    if t < 2.0:
        lam_ddot_true[k] = 0.0
    elif t < 3.5:
        lam_ddot_true[k] = 0.08   # rad/s^2 — 표적 기동 시작
    else:
        lam_ddot_true[k] = -0.04  # 기동 방향 변화

# 적분으로 각속도, 각도 생성
lam_dot_true[0] = 0.01  # 초기 각속도
for k in range(1, N):
    lam_dot_true[k] = lam_dot_true[k-1] + lam_ddot_true[k-1] * dt

lam_true = np.zeros(N)
for k in range(1, N):
    lam_true[k] = lam_true[k-1] + lam_dot_true[k-1] * dt

# 측정 잡음 (시커 각도 측정)
sigma_angle = np.radians(0.1)   # 0.1도 측정 잡음
z_arr = lam_true + np.random.randn(N) * sigma_angle

# ── α-β-γ 필터 파라미터 ───────────────────────────────────────────────────
# Benedict-Bordner 최적 조건 근사: α=0.5, β=0.1, γ=0.01
alpha = 0.50
beta  = 0.10
gamma = 0.01

# 상태: [각도, 각속도, 각가속도]
x_abg = np.array([z_arr[0], 0.0, 0.0])

lam_abg     = np.zeros(N)
ldot_abg    = np.zeros(N)
lddot_abg   = np.zeros(N)
lam_abg[0]  = x_abg[0]

for k in range(1, N):
    # Predict
    x_pred = np.array([
        x_abg[0] + x_abg[1]*dt + 0.5*x_abg[2]*dt**2,
        x_abg[1] + x_abg[2]*dt,
        x_abg[2]
    ])

    # Innovation
    nu = z_arr[k] - x_pred[0]

    # Update
    x_abg[0] = x_pred[0] + alpha * nu
    x_abg[1] = x_pred[1] + (beta  / dt)    * nu
    x_abg[2] = x_pred[2] + (2.0*gamma / dt**2) * nu

    lam_abg[k]   = x_abg[0]
    ldot_abg[k]  = x_abg[1]
    lddot_abg[k] = x_abg[2]

# ── 시각화 ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
fig.suptitle('Alpha-Beta-Gamma 필터: LOS 각도/각속도/각가속도 추정', fontsize=14, fontweight='bold')

# 각도
axes[0].plot(t_arr, np.degrees(lam_true), 'k-', lw=2, label='참값')
axes[0].scatter(t_arr[::5], np.degrees(z_arr[::5]), s=6, c='gray', alpha=0.4, label='측정값')
axes[0].plot(t_arr, np.degrees(lam_abg), 'r-', lw=1.5, label='α-β-γ 추정')
axes[0].set_ylabel('LOS 각도 (deg)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 각속도
axes[1].plot(t_arr, np.degrees(lam_dot_true), 'k-', lw=2, label='참값')
axes[1].plot(t_arr, np.degrees(ldot_abg), 'b-', lw=1.5, label='α-β-γ 추정')
axes[1].set_ylabel('LOS 각속도 (deg/s)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].axvspan(2.0, 3.5, alpha=0.1, color='orange', label='기동 구간')

# 각가속도
axes[2].plot(t_arr, np.degrees(lam_ddot_true), 'k-', lw=2, label='참값')
axes[2].plot(t_arr, np.degrees(lddot_abg), 'g-', lw=1.5, label='α-β-γ 추정')
axes[2].set_ylabel('LOS 각가속도 (deg/s²)')
axes[2].set_xlabel('시간 (s)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

rmse_lam  = np.sqrt(np.mean((lam_abg - lam_true)**2))
rmse_ldot = np.sqrt(np.mean((ldot_abg - lam_dot_true)**2))
print(f'각도 RMSE:    {np.degrees(rmse_lam):.4f} deg')
print(f'각속도 RMSE:  {np.degrees(rmse_ldot):.4f} deg/s')

## 3. Strapdown INS 기계화 (Mechanization)

### 관성항법의 원리

Strapdown INS는 IMU(자이로 + 가속도계)를 기체에 **직결**하여 동체 좌표계(Body Frame)에서 직접 측정합니다. (Platform INS와 달리 기계적 짐벌이 없음)

**좌표계 정의 (Siouris Ch.3):**
- **항법 좌표계 (n-frame):** NED (North-East-Down)
- **동체 좌표계 (b-frame):** x-전방, y-우측, z-하방
- **변환:** $\mathbf{C}^n_b$ (Body→NED, Direction Cosine Matrix)

---

### 기계화 방정식 (Mechanization Equations)

#### 1단계: 자세 갱신 (Attitude Update) — 쿼터니언

자이로 출력 $\boldsymbol{\omega}^b_{nb}$ → 회전 벡터 $\boldsymbol{\phi} = \boldsymbol{\omega}^b_{nb} \cdot \Delta t$

$$
\Delta\mathbf{q} = \begin{bmatrix} \cos(|\boldsymbol{\phi}|/2) \\ \sin(|\boldsymbol{\phi}|/2)\,\hat{\boldsymbol{\phi}} \end{bmatrix}
$$

$$
\mathbf{q}_{k} = \mathbf{q}_{k-1} \otimes \Delta\mathbf{q}
$$

#### 2단계: 속도 갱신 (Velocity Update)

가속도계 출력 $\mathbf{f}^b$ (specific force) → NED 변환 후 중력 보상:

$$
\dot{\mathbf{v}}^n = \mathbf{C}^n_b\,\mathbf{f}^b + \mathbf{g}^n
$$

여기서 $\mathbf{g}^n = [0, 0, +g]^\top$ (NED에서 중력은 아래 방향, z-Down 양수)

#### 3단계: 위치 갱신 (Position Update)

$$
\dot{\mathbf{p}}^n = \mathbf{v}^n
$$

---

### 관성항법의 특성

| 특성 | 설명 |
|------|------|
| **자율성** | 외부 신호 불필요 — 재밍 무관 |
| **단기 정확** | 짧은 비행 시간 동안 정밀 추적 |
| **장기 drift** | 오차가 **시간의 제곱**에 비례하여 누적 |
| **Drift 원인** | 자이로 bias → 자세 오차 → 속도/위치 오차 |

> **핵심:** 자이로 bias $b_g$가 있으면 자세 오차 $\psi \sim b_g \cdot t$, 속도 오차 $\delta v \sim g \cdot \psi \cdot t$, 위치 오차 $\delta p \sim \frac{1}{2} g \cdot b_g \cdot t^2$ — 2차 누적!

In [ ]:
"""
IMU 오차 모델 시뮬레이션

- 자이로: Gauss-Markov bias + 백색 잡음 (ARW)
- 가속도계: Gauss-Markov bias + 백색 잡음 (VRW)

Bar-Shalom Ch.11 / IMUModel 참조 구현
"""
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

dt      = 0.01    # 100 Hz IMU
T_total = 60.0    # 60초
N       = int(T_total / dt)
t_arr   = np.arange(N) * dt

# ── IMU 오차 파라미터 (전술급) ────────────────────────────────────────────
# 자이로
gyro_bias_std  = np.radians(5.0) / 3600.0   # 5 deg/hr → rad/s
gyro_noise_std = np.radians(0.5) / np.sqrt(3600.0)  # 0.5 deg/√hr → rad/√s
gyro_tau       = 300.0  # 상관 시간 (s)

# 가속도계
accel_bias_std  = 0.5e-3 * 9.81   # 0.5 mg → m/s^2
accel_noise_std = 0.1e-3 * 9.81   # 0.1 mg → m/s^2
accel_tau       = 600.0

# ── Gauss-Markov bias 시뮬레이션 ─────────────────────────────────────────
# b(k+1) = phi * b(k) + sigma_drive * w,  steady-state var = bias_std^2

def simulate_gauss_markov(N, dt, bias_std, tau, seed=None):
    """1축 Gauss-Markov 프로세스 시뮬레이션"""
    if seed is not None:
        rng = np.random.default_rng(seed)
    else:
        rng = np.random.default_rng()
    phi = np.exp(-dt / tau)
    sigma_drive = bias_std * np.sqrt(1.0 - phi**2)
    b = np.zeros(N)
    b[0] = rng.normal(0, bias_std)
    for k in range(1, N):
        b[k] = phi * b[k-1] + sigma_drive * rng.standard_normal()
    return b

# 자이로 x축 bias
gyro_bias_x = simulate_gauss_markov(N, dt, gyro_bias_std, gyro_tau, seed=1)

# 가속도계 x축 bias
accel_bias_x = simulate_gauss_markov(N, dt, accel_bias_std, accel_tau, seed=2)

# 백색 잡음 (1/√dt 스케일링)
noise_scale  = 1.0 / np.sqrt(dt)
gyro_noise   = gyro_noise_std  * noise_scale * np.random.randn(N)
accel_noise  = accel_noise_std * noise_scale * np.random.randn(N)

# 총 오차 = bias + 백색 잡음
gyro_error  = gyro_bias_x  + gyro_noise
accel_error = accel_bias_x + accel_noise

# ── 시각화 ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('IMU 오차 모델 (전술급 IMU)', fontsize=13, fontweight='bold')

# 자이로 bias
axes[0,0].plot(t_arr, np.degrees(gyro_bias_x)*3600, 'b-', lw=1)
axes[0,0].axhline(np.degrees(gyro_bias_std)*3600, color='r', ls='--', label=f'1σ = {np.degrees(gyro_bias_std)*3600:.1f} deg/hr')
axes[0,0].axhline(-np.degrees(gyro_bias_std)*3600, color='r', ls='--')
axes[0,0].set_title('자이로 Gauss-Markov Bias')
axes[0,0].set_ylabel('오차 (deg/hr)')
axes[0,0].set_xlabel('시간 (s)')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# 자이로 총 오차
axes[0,1].plot(t_arr, np.degrees(gyro_error)*3600, 'b-', lw=0.5, alpha=0.7)
axes[0,1].set_title('자이로 총 오차 (Bias + ARW)')
axes[0,1].set_ylabel('오차 (deg/hr)')
axes[0,1].set_xlabel('시간 (s)')
axes[0,1].grid(True, alpha=0.3)

# 가속도계 bias
axes[1,0].plot(t_arr, accel_bias_x * 1000 / 9.81, 'g-', lw=1)
axes[1,0].axhline(accel_bias_std*1000/9.81, color='r', ls='--', label=f'1σ = {accel_bias_std*1000/9.81:.3f} mg')
axes[1,0].axhline(-accel_bias_std*1000/9.81, color='r', ls='--')
axes[1,0].set_title('가속도계 Gauss-Markov Bias')
axes[1,0].set_ylabel('오차 (mg)')
axes[1,0].set_xlabel('시간 (s)')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# 가속도계 총 오차
axes[1,1].plot(t_arr, accel_error * 1000 / 9.81, 'g-', lw=0.5, alpha=0.7)
axes[1,1].set_title('가속도계 총 오차 (Bias + VRW)')
axes[1,1].set_ylabel('오차 (mg)')
axes[1,1].set_xlabel('시간 (s)')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('=== IMU 오차 파라미터 (전술급) ===')
print(f'자이로 bias 1σ:    {np.degrees(gyro_bias_std)*3600:.1f} deg/hr')
print(f'자이로 ARW:         {np.degrees(gyro_noise_std)*60:.3f} deg/√min')
print(f'가속도계 bias 1σ:  {accel_bias_std*1000/9.81:.3f} mg')
print(f'가속도계 VRW:       {accel_noise_std*1000/9.81:.4f} mg/√s')

In [ ]:
"""
Strapdown INS 구현 및 Drift 시뮬레이션

IMU 오차가 없는 경우 vs 있는 경우의 위치 오차 누적을 비교합니다.
자이로 bias → 자세 오차 → 중력 leakage → 속도 오차 → 위치 오차 (2차 누적)
"""
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# ── 쿼터니언 유틸리티 ─────────────────────────────────────────────────────
def quat_mult(q, r):
    """Hamilton product q ⊗ r (scalar-first)"""
    q0, q1, q2, q3 = q
    r0, r1, r2, r3 = r
    return np.array([
        q0*r0 - q1*r1 - q2*r2 - q3*r3,
        q0*r1 + q1*r0 + q2*r3 - q3*r2,
        q0*r2 - q1*r3 + q2*r0 + q3*r1,
        q0*r3 + q1*r2 - q2*r1 + q3*r0,
    ])

def quat_normalize(q):
    return q / np.linalg.norm(q)

def quat_to_dcm(q):
    """쿼터니언 → DCM (NED←Body, v_b = C_nb @ v_n)"""
    q0, q1, q2, q3 = q
    return np.array([
        [1-2*(q2**2+q3**2),  2*(q1*q2-q0*q3),  2*(q1*q3+q0*q2)],
        [2*(q1*q2+q0*q3),    1-2*(q1**2+q3**2), 2*(q2*q3-q0*q1)],
        [2*(q1*q3-q0*q2),    2*(q2*q3+q0*q1),   1-2*(q1**2+q2**2)],
    ])

def sins_propagate(pos, vel, quat, omega_b, f_b, dt, g=9.81):
    """Strapdown INS 한 스텝 전파 (플랫-지구, NED)"""
    # 1. 자세 갱신
    phi = omega_b * dt
    angle = np.linalg.norm(phi)
    if angle > 1e-10:
        axis = phi / angle
        dq = np.array([np.cos(angle/2), *(np.sin(angle/2)*axis)])
    else:
        dq = np.array([1.0, 0.5*phi[0], 0.5*phi[1], 0.5*phi[2]])
    dq = quat_normalize(dq)
    quat_new = quat_normalize(quat_mult(quat, dq))

    # 2. 속도 갱신 (C_nb: Body→NED)
    C_nb = quat_to_dcm(quat_new).T  # quat_to_dcm = C_nb (NED←Body)^T = C_bn
    g_ned = np.array([0.0, 0.0, g])
    vel_old = vel.copy()
    dv_ned = C_nb @ f_b * dt + g_ned * dt
    vel_new = vel + dv_ned

    # 3. 위치 갱신 (사다리꼴 적분)
    pos_new = pos + 0.5 * (vel_old + vel_new) * dt

    return pos_new, vel_new, quat_new

# ── 시뮬레이션 파라미터 ───────────────────────────────────────────────────
dt      = 0.01
T_total = 60.0
N       = int(T_total / dt)
t_arr   = np.arange(N) * dt
g       = 9.81

# 참값: 100 m/s 수평 등속 비행
v_true = np.array([100.0, 0.0, 0.0])  # NED: 북쪽으로 100 m/s

# 참 IMU 입력 (등속 = accel=0, 중력 제거된 specific force)
omega_true = np.zeros(3)
f_true = np.array([0.0, 0.0, 0.0])  # specific force = 0 (등속)

# 초기 상태
pos0  = np.zeros(3)
vel0  = v_true.copy()
quat0 = np.array([1.0, 0.0, 0.0, 0.0])  # 항등 쿼터니언 (NED=Body 초기)

# 자이로 bias 설정 (전술급)
gyro_bias = np.array([np.radians(5.0)/3600.0, 0.0, 0.0])  # x축 5 deg/hr

# ── 두 가지 시뮬레이션 ────────────────────────────────────────────────────
# A: 이상적 INS (오차 없음)
# B: 자이로 bias + 잡음 포함

pos_A, vel_A, quat_A = pos0.copy(), vel0.copy(), quat0.copy()
pos_B, vel_B, quat_B = pos0.copy(), vel0.copy(), quat0.copy()

pos_err_A = np.zeros(N)
pos_err_B = np.zeros(N)

noise_scale = 1.0 / np.sqrt(dt)
gyro_noise_std = np.radians(0.5) / np.sqrt(3600.0)

for k in range(N):
    # 참 위치
    pos_true_k = pos0 + v_true * t_arr[k]

    # A: 이상 IMU
    pos_A, vel_A, quat_A = sins_propagate(pos_A, vel_A, quat_A, omega_true, f_true, dt, g)
    pos_err_A[k] = np.linalg.norm(pos_A - pos_true_k)

    # B: 오차 포함 IMU
    omega_meas = omega_true + gyro_bias + gyro_noise_std * noise_scale * np.random.randn(3)
    f_meas     = f_true   # 가속도계 오차 제외 (자이로 효과만 보기 위함)
    pos_B, vel_B, quat_B = sins_propagate(pos_B, vel_B, quat_B, omega_meas, f_meas, dt, g)
    pos_err_B[k] = np.linalg.norm(pos_B - pos_true_k)

# 이론값: δp ≈ 0.5 * g * b_g * t^2 (Siouris)
b_g = gyro_bias[0]
pos_err_theory = 0.5 * g * b_g * t_arr**2

# ── 시각화 ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Strapdown INS Drift 분석', fontsize=13, fontweight='bold')

# 위치 오차 성장
axes[0].plot(t_arr, pos_err_A, 'g-', lw=1.5, label='이상 INS (오차 없음)')
axes[0].plot(t_arr, pos_err_B, 'r-', lw=1.5, label=f'오차 포함 (자이로 bias {np.degrees(b_g)*3600:.0f} deg/hr)')
axes[0].plot(t_arr, pos_err_theory, 'k--', lw=1, label='이론값 ½·g·b_g·t²')
axes[0].set_xlabel('시간 (s)')
axes[0].set_ylabel('위치 오차 (m)')
axes[0].set_title('위치 오차 성장 (자이로 bias 영향)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 로그 스케일로 2차 성장 확인
axes[1].semilogy(t_arr[1:], pos_err_B[1:], 'r-', lw=1.5)
axes[1].semilogy(t_arr[1:], pos_err_theory[1:], 'k--', lw=1, label='t² 이론선')
axes[1].set_xlabel('시간 (s)')
axes[1].set_ylabel('위치 오차 (m, 로그 스케일)')
axes[1].set_title('오차 누적 (로그 스케일 — 기울기 2 = 2차 성장)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'60초 후 INS 위치 오차: {pos_err_B[-1]:.2f} m')
print(f'이론 예측값:           {pos_err_theory[-1]:.2f} m')

## 4. GPS 측정 모델

GPS(Global Positioning System)는 유도탄 항법에서 INS의 drift를 주기적으로 보정하는 핵심 aiding sensor입니다.

### 측정 특성

| 파라미터 | 값 | 비고 |
|----------|-----|------|
| **위치 정확도** | $\sigma_{pos} \approx 5\text{–}10$ m (SEP) | Typical CEP; DGPS면 ~1m |
| **속도 정확도** | $\sigma_{vel} \approx 0.1$ m/s | 도플러 기반 |
| **업데이트 주기** | 1–10 Hz | 항법급: 10Hz |
| **지연** | ~0.1–1 s | 처리 지연 포함 |

### 측정 모델

$$
\mathbf{z}_{pos} = \mathbf{p}_{true} + \boldsymbol{\eta}_{pos}, \quad \boldsymbol{\eta}_{pos} \sim \mathcal{N}(\mathbf{0},\, \sigma^2_{pos}\mathbf{I}_3)
$$

$$
\mathbf{z}_{vel} = \mathbf{v}_{true} + \boldsymbol{\eta}_{vel}, \quad \boldsymbol{\eta}_{vel} \sim \mathcal{N}(\mathbf{0},\, \sigma^2_{vel}\mathbf{I}_3)
$$

### GPS Jamming 대응

유도탄 환경에서는 적의 **GPS jamming**이 현실적 위협입니다.
- Jamming 상황: GPS 측정 불가 → INS-only로 fallback → drift 급증
- 대안: TERCOM(지형대조), 광학 항법, 레이더 고도계, INS/GPS/레이더 복합

> **LIG Nex1 관련:** GPS 재밍 환경에서의 강건한 복합 항법은 능동 연구 분야입니다.

## 5. GPS-Aided INS: 15-State Error-State EKF

### 설계 철학: Indirect (Error-State) EKF

전체 상태(absolute state)를 추정하는 대신, INS가 계산한 항법 해와 참값 사이의 **오차(error)**만 추정합니다.

$$
\delta\mathbf{x} = \mathbf{x}_{true} - \mathbf{x}_{INS}
$$

**장점:**
- INS 비선형 역학은 INS가 직접 처리 → EKF는 소오차 선형 모델만 다룸
- 오차량이 작아 선형화 오차 작음
- 수치 안정성 우수

---

### 오차 상태 벡터 (15 States)

$$
\delta\mathbf{x} = \begin{bmatrix}
\delta\mathbf{p} \\ \delta\mathbf{v} \\ \boldsymbol{\psi} \\ \delta\mathbf{b}_a \\ \delta\mathbf{b}_g
\end{bmatrix}
\in \mathbb{R}^{15}
$$

| 상태 | 크기 | 의미 |
|------|------|------|
| $\delta\mathbf{p}$ | 3 | 위치 오차 (NED, m) |
| $\delta\mathbf{v}$ | 3 | 속도 오차 (NED, m/s) |
| $\boldsymbol{\psi}$ | 3 | 자세 소각도 오차 (rad) |
| $\delta\mathbf{b}_a$ | 3 | 가속도계 bias (m/s²) |
| $\delta\mathbf{b}_g$ | 3 | 자이로 bias (rad/s) |

---

### 연속시간 오차 상태 방정식 (F 행렬, 15×15)

$$
\dot{\delta\mathbf{x}} = \mathbf{F}\,\delta\mathbf{x} + \mathbf{G}\,\mathbf{w}
$$

$$
\mathbf{F} = \begin{bmatrix}
\mathbf{0} & \mathbf{I} & \mathbf{0} & \mathbf{0} & \mathbf{0} \\
\mathbf{0} & \mathbf{0} & -[\mathbf{f}^n\times] & \mathbf{C}^n_b & \mathbf{0} \\
\mathbf{0} & \mathbf{0} & \mathbf{0} & \mathbf{0} & -\mathbf{C}^n_b \\
\mathbf{0} & \mathbf{0} & \mathbf{0} & -\mathbf{I}/\tau_a & \mathbf{0} \\
\mathbf{0} & \mathbf{0} & \mathbf{0} & \mathbf{0} & -\mathbf{I}/\tau_g
\end{bmatrix}
$$

여기서 $[\mathbf{f}^n\times]$는 NED 프레임 specific force의 skew-symmetric 행렬.

---

### GPS 측정 행렬 (H 행렬)

GPS 위치+속도 동시 업데이트 (6-DOF):

$$
\mathbf{H}_{GPS} = \begin{bmatrix}
\mathbf{I}_3 & \mathbf{0} & \mathbf{0} & \mathbf{0} & \mathbf{0} \\
\mathbf{0} & \mathbf{I}_3 & \mathbf{0} & \mathbf{0} & \mathbf{0}
\end{bmatrix} \in \mathbb{R}^{6\times15}
$$

Innovation:
$$
\boldsymbol{\nu} = \begin{bmatrix} \mathbf{z}_{GPS,pos} - \mathbf{p}_{INS} \\ \mathbf{z}_{GPS,vel} - \mathbf{v}_{INS} \end{bmatrix}
$$

---

### INS 리셋 (Feedback Correction)

Kalman 업데이트 후 오차 추정치 $\hat{\delta\mathbf{x}}$를 INS에 직접 보정:

$$
\mathbf{p}_{INS} \leftarrow \mathbf{p}_{INS} + \hat{\delta\mathbf{p}}
$$
$$
\mathbf{v}_{INS} \leftarrow \mathbf{v}_{INS} + \hat{\delta\mathbf{v}}
$$
$$
\mathbf{q}_{INS} \leftarrow \delta\mathbf{q}(\boldsymbol{\psi}) \otimes \mathbf{q}_{INS}
$$

보정 후 오차 상태를 0으로 초기화합니다. (Groves Ch.14)

> **핵심 특성:** "INS의 단기 정확성 + GPS의 장기 안정성 결합" — INS가 GPS 업데이트 사이의 고주파 기동을 부드럽게 추적하고, GPS가 INS drift를 주기적으로 수정합니다.

In [ ]:
"""
15-State Error-State Navigation EKF 구현

NavKalmanFilter 참조 구현 (src/sensors/nav_kalman_filter.py 기반)
- Error state: [δp(3), δv(3), ψ(3), b_a(3), b_g(3)]
- 측정: GPS 위치+속도 (6-DOF)
- Joseph form 공분산 갱신
"""
import numpy as np

def skew(v):
    """3x3 skew-symmetric (cross-product) 행렬"""
    return np.array([
        [0.0,  -v[2],  v[1]],
        [v[2],   0.0, -v[0]],
        [-v[1],  v[0],  0.0],
    ])


class NavEKF15:
    """
    15-State Error-State EKF for GPS-aided INS.
    
    오차 상태: [δp(3), δv(3), ψ(3), b_a(3), b_g(3)]
    """
    N = 15
    POS = slice(0,3); VEL = slice(3,6); ATT = slice(6,9)
    BA  = slice(9,12); BG  = slice(12,15)

    def __init__(self, gyro_arw, accel_vrw, gyro_bias_std, accel_bias_std,
                 gyro_tau=300.0, accel_tau=600.0,
                 init_pos_sigma=10.0, init_vel_sigma=1.0, init_att_deg=1.0):
        self.gyro_arw       = gyro_arw
        self.accel_vrw      = accel_vrw
        self.gyro_bias_std  = gyro_bias_std
        self.accel_bias_std = accel_bias_std
        self.gyro_tau       = gyro_tau
        self.accel_tau      = accel_tau

        self.x = np.zeros(self.N)
        att_s = np.radians(init_att_deg)
        self.P = np.diag(np.concatenate([
            np.full(3, init_pos_sigma**2),
            np.full(3, init_vel_sigma**2),
            np.full(3, att_s**2),
            np.full(3, accel_bias_std**2),
            np.full(3, gyro_bias_std**2),
        ]))

    def predict(self, f_b, C_bn, dt):
        """오차 상태 전파 (Euler 이산화, Φ = I + F·dt)"""
        f_n = C_bn @ f_b   # specific force in NED

        F = np.zeros((self.N, self.N))
        F[self.POS, self.VEL] = np.eye(3)
        F[self.VEL, self.ATT] = -skew(f_n)
        F[self.VEL, self.BA]  =  C_bn
        F[self.ATT, self.BG]  = -C_bn
        F[self.BA,  self.BA]  = -np.eye(3) / self.accel_tau
        F[self.BG,  self.BG]  = -np.eye(3) / self.gyro_tau

        Phi = np.eye(self.N) + F * dt

        Q = np.zeros((self.N, self.N))
        Q[self.VEL, self.VEL] = self.accel_vrw**2  * dt * np.eye(3)
        Q[self.ATT, self.ATT] = self.gyro_arw**2   * dt * np.eye(3)
        Q[self.BA,  self.BA]  = (2*self.accel_bias_std**2/self.accel_tau*dt) * np.eye(3)
        Q[self.BG,  self.BG]  = (2*self.gyro_bias_std**2 /self.gyro_tau *dt) * np.eye(3)

        self.x = Phi @ self.x
        self.P = Phi @ self.P @ Phi.T + Q
        self.P = 0.5*(self.P + self.P.T)

    def correct_gps_posvel(self, innovation, R):
        """GPS 위치+속도 업데이트 (6-DOF, Joseph form)"""
        H = np.zeros((6, self.N))
        H[0:3, self.POS] = np.eye(3)
        H[3:6, self.VEL] = np.eye(3)

        S   = H @ self.P @ H.T + R
        K   = self.P @ H.T @ np.linalg.inv(S)

        dx = K @ innovation
        self.x += dx

        I   = np.eye(self.N)
        IKH = I - K @ H
        self.P = IKH @ self.P @ IKH.T + K @ R @ K.T
        self.P = 0.5*(self.P + self.P.T)

        return self.x.copy()

    def reset_error_state(self):
        self.x[:] = 0.0

print('NavEKF15 클래스 정의 완료')
print(f'오차 상태 차원: {NavEKF15.N}')
print('  δp [0:3]  위치 오차 (NED, m)')
print('  δv [3:6]  속도 오차 (NED, m/s)')
print('  ψ  [6:9]  자세 오차 (rad)')
print('  b_a[9:12] 가속도계 bias (m/s²)')
print('  b_g[12:15] 자이로 bias (rad/s)')

In [ ]:
"""
시뮬레이션: 60초 비행 중 GPS-Aided INS vs INS-Only

시나리오:
- 초기 100 m/s 등속 수평 비행 → 30초 후 기동 (10g 횡 가속도 5초)
- GPS 업데이트: 1 Hz (1초마다)
- INS-only: 자이로 bias → 위치 오차 2차 누적
- GPS-aided: drift가 bounded됨을 확인
"""
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(123)

# ── 시뮬레이션 파라미터 ───────────────────────────────────────────────────
dt_imu  = 0.01    # 100 Hz IMU
dt_gps  = 1.0     # 1 Hz GPS
T_total = 60.0
N_imu   = int(T_total / dt_imu)
t_arr   = np.arange(N_imu) * dt_imu
g       = 9.81

# IMU 오차 파라미터 (전술급)
gyro_bias_std   = np.radians(5.0) / 3600.0
accel_bias_std  = 0.5e-3 * 9.81
gyro_arw        = np.radians(0.5) / np.sqrt(3600.0)
accel_vrw       = 0.1e-3 * 9.81
gyro_tau        = 300.0
accel_tau       = 600.0

# GPS 측정 잡음
gps_pos_sigma = 5.0    # m
gps_vel_sigma = 0.1    # m/s
R_gps = np.diag(np.concatenate([np.full(3, gps_pos_sigma**2),
                                  np.full(3, gps_vel_sigma**2)]))

# ── 참 궤적 생성 ──────────────────────────────────────────────────────────
# 등속 → 기동 (30~35s: 10g 횡 가속도)
v_x = 100.0  # m/s 북방향
accel_cmd = np.zeros((N_imu, 3))   # NED
for k in range(N_imu):
    t = t_arr[k]
    if 30.0 <= t < 35.0:
        accel_cmd[k, 1] = 10.0 * g   # 동쪽 방향 10g 기동

pos_true = np.zeros((N_imu, 3))
vel_true = np.zeros((N_imu, 3))
vel_true[0] = [v_x, 0.0, 0.0]

for k in range(1, N_imu):
    vel_true[k] = vel_true[k-1] + accel_cmd[k-1] * dt_imu
    pos_true[k] = pos_true[k-1] + 0.5*(vel_true[k-1]+vel_true[k])*dt_imu

# ── IMU 오차 생성 ─────────────────────────────────────────────────────────
# 초기 bias 드로우
gyro_bias  = np.random.randn(3) * gyro_bias_std
accel_bias = np.random.randn(3) * accel_bias_std

gyro_meas_arr  = np.zeros((N_imu, 3))
accel_meas_arr = np.zeros((N_imu, 3))
ns = 1.0 / np.sqrt(dt_imu)

phi_g = np.exp(-dt_imu / gyro_tau)
phi_a = np.exp(-dt_imu / accel_tau)
sd_g  = gyro_bias_std  * np.sqrt(1.0 - phi_g**2)
sd_a  = accel_bias_std * np.sqrt(1.0 - phi_a**2)

for k in range(N_imu):
    # true IMU 입력: specific force = accel - gravity (NED), omega = 0 (등속/기동)
    f_true = accel_cmd[k] - np.array([0.0, 0.0, g])  # specific force in NED
    omega_true = np.zeros(3)

    # bias update
    gyro_bias  = phi_g * gyro_bias  + sd_g * np.random.randn(3)
    accel_bias = phi_a * accel_bias + sd_a * np.random.randn(3)

    gyro_meas_arr[k]  = omega_true + gyro_bias  + gyro_arw  * ns * np.random.randn(3)
    accel_meas_arr[k] = f_true     + accel_bias + accel_vrw * ns * np.random.randn(3)

# ── 항법 루프 ─────────────────────────────────────────────────────────────
def quat_mult(q, r):
    q0,q1,q2,q3 = q; r0,r1,r2,r3 = r
    return np.array([q0*r0-q1*r1-q2*r2-q3*r3, q0*r1+q1*r0+q2*r3-q3*r2,
                     q0*r2-q1*r3+q2*r0+q3*r1, q0*r3+q1*r2-q2*r1+q3*r0])

def quat_norm(q): return q / np.linalg.norm(q)

def quat_to_dcm_bn(q):
    """Body→NED DCM (C_bn = C_nb^T)"""
    q0,q1,q2,q3 = q
    C_nb = np.array([
        [1-2*(q2**2+q3**2), 2*(q1*q2-q0*q3), 2*(q1*q3+q0*q2)],
        [2*(q1*q2+q0*q3), 1-2*(q1**2+q3**2), 2*(q2*q3-q0*q1)],
        [2*(q1*q3-q0*q2), 2*(q2*q3+q0*q1), 1-2*(q1**2+q2**2)],
    ])
    return C_nb.T  # Body→NED

# 초기화
pos_ins  = np.zeros(3)
vel_ins  = vel_true[0].copy()
quat_ins = np.array([1.0, 0.0, 0.0, 0.0])

# INS-only 별도 저장
pos_ins_only  = np.zeros(3)
vel_ins_only  = vel_true[0].copy()
quat_ins_only = np.array([1.0, 0.0, 0.0, 0.0])

# EKF 초기화
ekf = NavEKF15(
    gyro_arw=gyro_arw, accel_vrw=accel_vrw,
    gyro_bias_std=gyro_bias_std, accel_bias_std=accel_bias_std,
    gyro_tau=gyro_tau, accel_tau=accel_tau,
    init_pos_sigma=gps_pos_sigma, init_vel_sigma=gps_vel_sigma
)

# 결과 저장
pos_err_ins    = np.zeros(N_imu)   # INS-only 위치 오차
pos_err_aided  = np.zeros(N_imu)   # GPS-aided 위치 오차
vel_err_ins    = np.zeros(N_imu)
vel_err_aided  = np.zeros(N_imu)

t_gps_next = dt_gps
gps_update_times = []

for k in range(N_imu):
    t = t_arr[k]
    omega_m = gyro_meas_arr[k]
    f_m     = accel_meas_arr[k]

    # ── INS 전파 (aided) ───────────────────────────────────────────────
    phi = omega_m * dt_imu
    angle = np.linalg.norm(phi)
    if angle > 1e-10:
        dq = np.array([np.cos(angle/2), *(np.sin(angle/2)*phi/angle)])
    else:
        dq = np.array([1.0, 0.5*phi[0], 0.5*phi[1], 0.5*phi[2]])
    quat_ins = quat_norm(quat_mult(quat_ins, dq))
    C_bn = quat_to_dcm_bn(quat_ins)
    g_ned = np.array([0.0, 0.0, g])
    vel_old = vel_ins.copy()
    vel_ins = vel_ins + C_bn @ f_m * dt_imu + g_ned * dt_imu
    pos_ins = pos_ins + 0.5*(vel_old + vel_ins) * dt_imu

    # ── INS 전파 (only) ────────────────────────────────────────────────
    quat_ins_only = quat_norm(quat_mult(quat_ins_only, dq))
    C_bn_o = quat_to_dcm_bn(quat_ins_only)
    vel_old_o = vel_ins_only.copy()
    vel_ins_only = vel_ins_only + C_bn_o @ f_m * dt_imu + g_ned * dt_imu
    pos_ins_only = pos_ins_only + 0.5*(vel_old_o + vel_ins_only) * dt_imu

    # ── EKF Predict ────────────────────────────────────────────────────
    ekf.predict(f_m, C_bn, dt_imu)

    # ── GPS Update ─────────────────────────────────────────────────────
    if t >= t_gps_next - 1e-9:
        gps_pos = pos_true[k] + np.random.randn(3) * gps_pos_sigma
        gps_vel = vel_true[k] + np.random.randn(3) * gps_vel_sigma

        inn = np.concatenate([gps_pos - pos_ins, gps_vel - vel_ins])
        dx = ekf.correct_gps_posvel(inn, R_gps)

        # INS 리셋
        pos_ins += dx[0:3]
        vel_ins += dx[3:6]
        psi = dx[6:9]
        half = 0.5 * psi
        dq_psi = quat_norm(np.array([1.0, half[0], half[1], half[2]]))
        quat_ins = quat_norm(quat_mult(dq_psi, quat_ins))
        ekf.reset_error_state()

        t_gps_next += dt_gps
        gps_update_times.append(t)

    pos_err_ins[k]   = np.linalg.norm(pos_ins_only - pos_true[k])
    pos_err_aided[k] = np.linalg.norm(pos_ins      - pos_true[k])
    vel_err_ins[k]   = np.linalg.norm(vel_ins_only - vel_true[k])
    vel_err_aided[k] = np.linalg.norm(vel_ins      - vel_true[k])

# ── 시각화 ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
fig.suptitle('GPS-Aided INS vs INS-Only: 60초 비행 시뮬레이션', fontsize=13, fontweight='bold')

# 위치 오차
ax1 = axes[0]
ax1.plot(t_arr, pos_err_ins,   'r-', lw=1.5, label='INS-only')
ax1.plot(t_arr, pos_err_aided, 'b-', lw=1.5, label='GPS-Aided INS (1 Hz GPS)')
for tg in gps_update_times:
    ax1.axvline(tg, color='green', alpha=0.2, linewidth=0.8)
ax1.axvspan(30, 35, alpha=0.1, color='orange', label='기동 구간 (10g)')
ax1.set_ylabel('위치 오차 (m)')
ax1.set_title('위치 오차 비교 (녹색 수직선: GPS 업데이트)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 속도 오차
ax2 = axes[1]
ax2.plot(t_arr, vel_err_ins,   'r-', lw=1.5, label='INS-only')
ax2.plot(t_arr, vel_err_aided, 'b-', lw=1.5, label='GPS-Aided INS')
ax2.axvspan(30, 35, alpha=0.1, color='orange')
ax2.set_xlabel('시간 (s)')
ax2.set_ylabel('속도 오차 (m/s)')
ax2.set_title('속도 오차 비교')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('=== 최종 성능 비교 (60초 비행) ===')
print(f'INS-only   위치 오차: {pos_err_ins[-1]:.2f} m')
print(f'GPS-aided  위치 오차: {pos_err_aided[-1]:.2f} m')
print(f'GPS-aided  속도 오차: {vel_err_aided[-1]:.3f} m/s')
print(f'\nGPS 보정으로 drift가 bounded됨을 확인!')

In [ ]:
"""
Covariance 분석: P(t) 대각 원소의 시간 변화

GPS 업데이트마다 covariance가 감소(reset)되는 것을 시각적으로 확인합니다.
이것이 필터의 "confidence" 변화를 보여줍니다.
"""
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# ── 파라미터 (이전 셀과 동일) ─────────────────────────────────────────────
dt_imu  = 0.01
dt_gps  = 1.0
T_total = 60.0
N_imu   = int(T_total / dt_imu)
t_arr   = np.arange(N_imu) * dt_imu

gyro_bias_std   = np.radians(5.0) / 3600.0
accel_bias_std  = 0.5e-3 * 9.81
gyro_arw        = np.radians(0.5) / np.sqrt(3600.0)
accel_vrw       = 0.1e-3 * 9.81
gps_pos_sigma   = 5.0
gps_vel_sigma   = 0.1
R_gps = np.diag(np.concatenate([np.full(3, gps_pos_sigma**2),
                                  np.full(3, gps_vel_sigma**2)]))

# EKF 초기화
ekf2 = NavEKF15(
    gyro_arw=gyro_arw, accel_vrw=accel_vrw,
    gyro_bias_std=gyro_bias_std, accel_bias_std=accel_bias_std,
    init_pos_sigma=gps_pos_sigma, init_vel_sigma=gps_vel_sigma
)

# 명목 DCM (단위행렬 — 수평 비행 가정)
C_bn_nom  = np.eye(3)
f_b_nom   = np.array([0.0, 0.0, 0.0])   # 등속

# covariance 기록 (다운샘플: 10Hz로)
record_dt = 0.1
record_N  = int(T_total / record_dt)
t_rec     = np.arange(record_N) * record_dt

P_pos_std  = np.zeros((record_N, 3))  # 위치 sigma [N, E, D]
P_vel_std  = np.zeros((record_N, 3))  # 속도 sigma
P_att_std  = np.zeros((record_N, 3))  # 자세 sigma (deg)
P_bg_std   = np.zeros((record_N, 3))  # 자이로 bias sigma (deg/hr)

rec_idx  = 0
t_gps_next = dt_gps
gps_times  = []

for k in range(N_imu):
    t = t_arr[k]

    ekf2.predict(f_b_nom, C_bn_nom, dt_imu)

    # GPS 업데이트 (이상적 — innovation = 0, GPS 잡음만)
    if t >= t_gps_next - 1e-9:
        inn = np.random.randn(6) * np.array([gps_pos_sigma]*3 + [gps_vel_sigma]*3)
        ekf2.correct_gps_posvel(inn, R_gps)
        ekf2.reset_error_state()
        t_gps_next += dt_gps
        gps_times.append(t)

    # 기록 (다운샘플)
    if rec_idx < record_N and abs(t - t_rec[rec_idx]) < dt_imu/2:
        P_diag = np.sqrt(np.maximum(np.diag(ekf2.P), 0.0))
        P_pos_std[rec_idx] = P_diag[0:3]
        P_vel_std[rec_idx] = P_diag[3:6]
        P_att_std[rec_idx] = np.degrees(P_diag[6:9])
        P_bg_std[rec_idx]  = np.degrees(P_diag[12:15]) * 3600  # deg/hr
        rec_idx += 1

# ── 시각화 ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Covariance 분석: GPS 업데이트 효과\n(GPS update마다 covariance가 감소됨을 확인)',
             fontsize=13, fontweight='bold')

labels_ned = ['North (N)', 'East (E)', 'Down (D)']
colors_ned = ['b', 'g', 'r']

# 위치 sigma
ax = axes[0, 0]
for i in range(3):
    ax.plot(t_rec, P_pos_std[:, i], color=colors_ned[i], lw=1.2, label=labels_ned[i])
for tg in gps_times[::5]:
    ax.axvline(tg, color='orange', alpha=0.3, lw=0.8)
ax.set_title('위치 1σ (Position Uncertainty)')
ax.set_ylabel('σ (m)')
ax.set_xlabel('시간 (s)')
ax.legend()
ax.grid(True, alpha=0.3)

# 속도 sigma
ax = axes[0, 1]
for i in range(3):
    ax.plot(t_rec, P_vel_std[:, i], color=colors_ned[i], lw=1.2, label=labels_ned[i])
for tg in gps_times[::5]:
    ax.axvline(tg, color='orange', alpha=0.3, lw=0.8)
ax.set_title('속도 1σ (Velocity Uncertainty)')
ax.set_ylabel('σ (m/s)')
ax.set_xlabel('시간 (s)')
ax.legend()
ax.grid(True, alpha=0.3)

# 자세 sigma
ax = axes[1, 0]
for i in range(3):
    ax.plot(t_rec, P_att_std[:, i], color=colors_ned[i], lw=1.2, label=labels_ned[i])
for tg in gps_times[::5]:
    ax.axvline(tg, color='orange', alpha=0.3, lw=0.8)
ax.set_title('자세 오차 1σ (Attitude Uncertainty)')
ax.set_ylabel('σ (deg)')
ax.set_xlabel('시간 (s)')
ax.legend()
ax.grid(True, alpha=0.3)

# 자이로 bias sigma
ax = axes[1, 1]
for i in range(3):
    ax.plot(t_rec, P_bg_std[:, i], color=colors_ned[i], lw=1.2, label=f'축 {i+1}')
ax.axhline(np.degrees(gyro_bias_std)*3600, color='k', ls='--', lw=1,
           label=f'Prior 1σ ({np.degrees(gyro_bias_std)*3600:.1f} deg/hr)')
ax.set_title('자이로 Bias 불확실도 1σ')
ax.set_ylabel('σ (deg/hr)')
ax.set_xlabel('시간 (s)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('=== Covariance 분석 결과 ===')
print(f'초기 위치 1σ:   {P_pos_std[0, 0]:.2f} m  →  정상상태: ~{np.mean(P_pos_std[-10:, 0]):.2f} m')
print(f'초기 속도 1σ:   {P_vel_std[0, 0]:.3f} m/s  →  정상상태: ~{np.mean(P_vel_std[-10:, 0]):.3f} m/s')
print(f'\n* GPS update마다 위치/속도 covariance가 GPS 측정 잡음 수준으로 reset됨')
print(f'* 자세/bias covariance는 GPS만으로는 완전 추정 불가 (간접 관측)')

## 6. 정리 (Summary)

---

### INS-Only vs GPS-Aided 비교표

| 항목 | INS-Only | GPS-Aided INS |
|------|----------|---------------|
| **위치 정확도** | 오차 ∝ $t^2$ (발산) | 오차 bounded (~GPS 정확도) |
| **단기 정확도** | 우수 (고속 갱신) | 우수 (INS 유지) |
| **장기 정확도** | 불량 (drift) | 양호 (GPS 보정) |
| **재밍 내성** | 완전 자율 | GPS 재밍 취약 |
| **갱신 주기** | 100+ Hz | 1–10 Hz (GPS 주기) |
| **계산 부담** | 낮음 | 중간 (EKF 15×15) |
| **자세 추정** | 자이로 적분 | 개선 (가속도계 보조) |

---

### 이 노트북에서 배운 핵심

1. **Kalman Filter** — 측정 잡음과 모델 불확실도를 확률적으로 최적 융합
   - Joseph form은 수치 안정성을 보장하는 필수 구현
   - Innovation 크기로 필터 건강도 모니터링 가능 (3-sigma gate)

2. **Alpha-Beta-Gamma** — 정상상태 Kalman의 고정게인 근사
   - LOS rate 추정 등 실시간 고속 처리에 유리
   - 기동 표적에는 γ 항 필요

3. **Strapdown INS** — 쿼터니언 적분으로 자세 추적, 자이로 bias → 위치 오차 2차 누적
   - 60초 비행에서 5 deg/hr 자이로 bias → 수십~수백 m 위치 오차

4. **GPS-Aided INS (15-state EKF)** — Error-state 공식으로 INS + GPS 최적 융합
   - GPS 업데이트마다 covariance reset → drift bounded
   - 자이로 bias도 점진적으로 추정 가능 (관찰가능성 분석 필요)

---

### 실전에서 가장 어려운 부분: Q, R 튜닝

> **"칼만 필터 이론은 단순하다. 어려운 건 Q와 R을 어떻게 설정하느냐다."**

- **Q 과소 설정:** 모델을 과도 신뢰 → 기동에 둔감, 편향 발생
- **Q 과대 설정:** 잡음에 과민 → 추정값 진동
- **R 과소 설정:** 측정을 과도 신뢰 → 외란에 취약
- **R 과대 설정:** 측정 무시 → GPS 보정 효과 감소

실전 접근법:
1. 센서 데이터시트 기반 초기값 설정
2. Allan Variance 분석으로 IMU 잡음 정확 측정
3. Monte Carlo 시뮬레이션으로 성능 검증
4. HILS(Hardware-in-the-Loop) 환경에서 최종 검증

---

### GPS Jamming 상황 고려 — 대안 항법

| 대안 항법 | 원리 | 특징 |
|-----------|------|------|
| **TERCOM** | 지형 고도 대조 | 지형 DB 필요, 해상 불가 |
| **DSMAC** | 광학 영상 대조 | 야간/악천후 취약 |
| **레이더 고도계** | 고도 측정 | 고도만 보정 |
| **별 추적기** | 성도 대조 | 주간 불가 |
| **INS/GPS/레이더 복합** | 다중 융합 | 최고 강건성 |

---

### 다음 단계

- **비선형 필터:** Unscented KF (UKF), Particle Filter — 고기동 표적 추적
- **관찰가능성 분석:** 어떤 기동이 bias 추정을 개선하는가?
- **고장 검출:** Innovation 모니터링, Chi-squared test
- **실전 연계:** LIG Nex1 HILS 환경에서의 항법 필터 검증 절차